In [24]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
df=pd.read_csv('../data/interim/listings_parsed_v6.csv')

In [26]:
print(len(df))
print(df["group_coarse"].nunique()) 

183490
22843


In [27]:
print(df[["Price_yen", "area_m2", "age_years", "floor_min",
          "stories_above", "walk_min_nearest", "municipality"]].isna().sum())
print(df[["Price_yen", "area_m2", "age_years", "floor_min",
          "stories_above", "walk_min_nearest"]].dtypes)

Price_yen              0
area_m2                0
age_years              0
floor_min           1004
stories_above          0
walk_min_nearest    1665
municipality           0
dtype: int64
Price_yen             int64
area_m2             float64
age_years             int64
floor_min           float64
stories_above         int64
walk_min_nearest    float64
dtype: object


In [28]:
# Run DummyRegressor(strategy="median") through evaluate_model twice:
#   1. KFold(n_splits=5, shuffle=True, random_state=0), groups=None
#   2. GroupKFold(n_splits=5), groups=df["group_coarse"]
# X = df[["area_m2"]] (the dummy ignores it), y = df["Price_yen"]
# Print the summary of each as a small table: split name, mean MAE, mean RMSE, mean R2, std MAE.
from src.evaluate import evaluate_model

dummy = DummyRegressor(strategy="median")

# First evaluation: KFold
X = df[["area_m2"]]
y = df["Price_yen"]
kf = KFold(n_splits=5, shuffle=True, random_state=0)
fold_df1, summary1 = evaluate_model(dummy, X, y, kf)

# Second evaluation: GroupKFold
gkf = GroupKFold(n_splits=5)
fold_df2, summary2 = evaluate_model(dummy, X, y, gkf, groups=df["group_coarse"])

# Print the summaries as tables
print("KFold Results:")
print(f"Split: {summary1['mae_mean']:.2f} ± {summary1['mae_std']:.2f}")
print(f"MAE: {summary1['mae_mean']:.2f} ± {summary1['mae_std']:.2f}")
print(f"RMSE: {summary1['rmse_mean']:.2f} ± {summary1['rmse_std']:.2f}")
print(f"R2: {summary1['r2_mean']:.2f} ± {summary1['r2_std']:.2f}")

print("\nGroupKFold Results:")

print(f"MAE: {summary2['mae_mean']:.2f} ± {summary2['mae_std']:.2f}")
print(f"RMSE: {summary2['rmse_mean']:.2f} ± {summary2['rmse_std']:.2f}")
print(f"R2: {summary2['r2_mean']:.2f} ± {summary2['r2_std']:.2f}")

KFold Results:
Split: 37946.12 ± 362.48
MAE: 37946.12 ± 362.48
RMSE: 79368.36 ± 2898.45
R2: -0.05 ± 0.00

GroupKFold Results:
MAE: 37948.14 ± 947.67
RMSE: 79222.83 ± 5787.76
R2: -0.05 ± 0.00


In [33]:
# Run MedianByGroup(column="municipality") (from src.models import MedianByGroup)
# through evaluate_model twice:
#   1. KFold(n_splits=5, shuffle=True, random_state=0), groups=None
#   2. GroupKFold(n_splits=5), groups=df["group_coarse"]
# X = df[["municipality"]], y = df["Price_yen"]
# Print one small table with columns: split name, mean MAE, mean RMSE, mean R2, std MAE.
# Print the split name from a separate variable; do not reuse a metric variable for it.
# Do not modify evaluate_model or MedianByGroup.

from src.models import MedianByGroup
median_by_muni = MedianByGroup(column="municipality")

X=df[["municipality"]]
y=df["Price_yen"]

kf = KFold(n_splits=5, shuffle=True, random_state=0)
fold_df1, summary1 = evaluate_model(median_by_muni, X, y, kf)

gkf = GroupKFold(n_splits=5)
fold_df2, summary2 = evaluate_model(median_by_muni, X, y, gkf, groups=df["group_coarse"])

print("KFold Results:")
print(f"MAE: {summary1['mae_mean']:.2f} ± {summary1['mae_std']:.2f}")
print(f"RMSE: {summary1['rmse_mean']:.2f} ± {summary1['rmse_std']:.2f}")
print(f"R2: {summary1['r2_mean']:.2f} ± {summary1['r2_std']:.2f}")

print("\nGroupKFold Results:")
print(f"MAE: {summary2['mae_mean']:.2f} ± {summary2['mae_std']:.2f}")
print(f"RMSE: {summary2['rmse_mean']:.2f} ± {summary2['rmse_std']:.2f}")
print(f"R2: {summary2['r2_mean']:.2f} ± {summary2['r2_std']:.2f}")





KFold Results:
MAE: 32059.39 ± 280.21
RMSE: 71371.08 ± 2893.81
R2: 0.15 ± 0.01

GroupKFold Results:
MAE: 32185.54 ± 830.65
RMSE: 71298.46 ± 5686.31
R2: 0.15 ± 0.01


In [35]:
# X = df[numeric cols + ["municipality"]], y = df["Price_yen"]
# Run through evaluate_model:
#   1. KFold(n_splits=5, shuffle=True, random_state=0), groups=None
#   2. GroupKFold(n_splits=5), groups=df["group_coarse"]   <- must pass groups=
# Print one small table: split name, mean MAE, mean RMSE, mean R2, std MAE.
# Print the split name from a separate variable.
# Do not fit the imputer, scaler or encoder outside the pipeline.
# Do not modify evaluate_model.

from src.models import linear_pipe

X = df[["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest", "municipality"]]
y = df["Price_yen"]

kf = KFold(n_splits=5, shuffle=True, random_state=0)
fold_df1, summary1 = evaluate_model(linear_pipe, X, y, kf)

gkf = GroupKFold(n_splits=5)
fold_df2, summary2 = evaluate_model(linear_pipe, X, y, gkf, groups=df["group_coarse"])

print("KFold Results:")
print(f"MAE: {summary1['mae_mean']:.2f} ± {summary1['mae_std']:.2f}")
print(f"RMSE: {summary1['rmse_mean']:.2f} ± {summary1['rmse_std']:.2f}")
print(f"R2: {summary1['r2_mean']:.2f} ± {summary1['r2_std']:.2f}")

print("\nGroupKFold Results:")
print(f"MAE: {summary2['mae_mean']:.2f} ± {summary2['mae_std']:.2f}")
print(f"RMSE: {summary2['rmse_mean']:.2f} ± {summary2['rmse_std']:.2f}")
print(f"R2: {summary2['r2_mean']:.2f} ± {summary2['r2_std']:.2f}")


KFold Results:
MAE: 20188.11 ± 100.88
RMSE: 42455.49 ± 2127.24
R2: 0.70 ± 0.01

GroupKFold Results:
MAE: 20275.99 ± 498.63
RMSE: 42478.00 ± 4027.62
R2: 0.70 ± 0.02


In [39]:
# For each of rf_pipe and xgb_pipe, run evaluate_model twice:
#   1. KFold(n_splits=5, shuffle=True, random_state=0), groups=None
#   2. GroupKFold(n_splits=5), groups=df["group_coarse"]   <- must pass groups=
# X = df[numeric cols + ["municipality"]], y = df["Price_yen"]
# Print one small table: model name, split name, mean MAE, mean RMSE, mean R2, std MAE.
# Take names from separate variables. Do not modify evaluate_model.

from src.models import rf_pipe, xgb_pipe

X=df[["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest", "municipality"]]
y=df["Price_yen"]

kf = KFold(n_splits=5, shuffle=True, random_state=0)
fold_df1_rf, summary1_rf = evaluate_model(rf_pipe, X, y, kf)
fold_df1_xgb, summary1_xgb = evaluate_model(xgb_pipe, X, y, kf) 

gkf = GroupKFold(n_splits=5)
fold_df2_rf, summary2_rf = evaluate_model(rf_pipe, X, y, gkf, groups=df["group_coarse"])
fold_df2_xgb, summary2_xgb = evaluate_model(xgb_pipe, X, y, gkf, groups=df["group_coarse"])

print("Random Forest Results:")
print("KFold Results:")
print(f"MAE: {summary1_rf['mae_mean']:.2f} ± {summary1_rf['mae_std']:.2f}")
print(f"RMSE: {summary1_rf['rmse_mean']:.2f} ± {summary1_rf['rmse_std']:.2f}")
print(f"R2: {summary1_rf['r2_mean']:.2f} ± {summary1_rf['r2_std']:.2f}")

print("\nGroupKFold Results:")
print(f"MAE: {summary2_rf['mae_mean']:.2f}  ± {summary2_rf['mae_std']:.2f}")
print(f"RMSE: {summary2_rf['rmse_mean']:.2f} ± {summary2_rf['rmse_std']:.2f}")
print(f"R2: {summary2_rf['r2_mean']:.2f} ± {summary2_rf['r2_std']:.2f}")

print("\nXGBoost Results:")
print("KFold Results:")
print(f"MAE: {summary1_xgb['mae_mean']:.2f} ± {summary1_xgb['mae_std']:.2f}")
print(f"RMSE: {summary1_xgb['rmse_mean']:.2f} ± {summary1_xgb['rmse_std']:.2f}")
print(f"R2: {summary1_xgb['r2_mean']:.2f} ± {summary1_xgb['r2_std']:.2f}")

print("\nGroupKFold Results:")
print(f"MAE: {summary2_xgb['mae_mean']:.2f}  ± {summary2_xgb['mae_std']:.2f}")
print(f"RMSE: {summary2_xgb['rmse_mean']:.2f} ± {summary2_xgb['rmse_std']:.2f}")
print(f"R2: {summary2_xgb['r2_mean']:.2f} ± {summary2_xgb['r2_std']:.2f}")




c:\Users\priya\Desktop\data science try\tokyo-rental-intelligence\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [15:17:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Random Forest Results:
KFold Results:
MAE: 7240.10 ± 92.06
RMSE: 23680.52 ± 2525.64
R2: 0.91 ± 0.02

GroupKFold Results:
MAE: 12107.93  ± 243.21
RMSE: 29142.34 ± 3060.61
R2: 0.86 ± 0.01

XGBoost Results:
KFold Results:
MAE: 9690.56 ± 136.05
RMSE: 27879.35 ± 2220.21
R2: 0.87 ± 0.01

GroupKFold Results:
MAE: 11302.69  ± 200.05
RMSE: 32264.15 ± 4476.08
R2: 0.83 ± 0.02
